# Model 2

Key difference from Model 1:
  - Tabular features (NDVI, Height) are embedded and fused with
    image features before the regression head.
  - This gives the model the physical signal that images alone can't provide

In [13]:
import os
import numpy as np
import pandas as pd
from PIL import Image
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
import matplotlib.pyplot as plt
import joblib  # for saving the scaler

class Config:
    # paths
    CSV_PATH        = "train.csv"
    IMAGE_DIR       = ""
    SAVE_PATH       = "model2_best.pth"
    SCALER_PATH     = "model2_scaler.pkl"   # save scaler for inference

    # targets
    # TARGETS = ["Dry_Green_g", "Dry_Dead_g", "Dry_Clover_g", "GDM_g", "Dry_Total_g"]
    TARGETS = ["Dry_Green_g", "GDM_g", "Dry_Total_g"]
    

    # tabular features
    TABULAR_FEATURES = ["Pre_GSHH_NDVI", "Height_Ave_cm"]

    # training
    IMAGE_SIZE   = 224
    BATCH_SIZE   = 16 #16
    NUM_EPOCHS   = 150 #60        # slightly more epochs — fusion model benefits from it
    LR           = 1e-4
    WEIGHT_DECAY = 1e-4
    VAL_SPLIT    = 0.2
    SEED         = 42

    # model
    BACKBONE     = 'resnet34' #"efficientnet_b0"   # or "resnet50", "resnet34"
    PRETRAINED   = True
    DROPOUT      = 0.3
    TAB_HIDDEN   = 256 #32        # hidden size for tabular embedding branch

    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

In [14]:
def prepare_data(csv_path, targets, tab_features, val_split=0.2, seed=42):
    """
    Pivots long-format CSV to wide format (one row per image),
    then fits a StandardScaler on the tabular features of the training set.
    """
    df = pd.read_csv(csv_path)

    # pivot targets to wide format
    df_wide = df.pivot_table(
        index=["image_path"] + tab_features,
        columns="target_name",
        values="target"
    ).reset_index()
    df_wide.columns.name = None

    # keep only targets we care about
    available = [t for t in targets if t in df_wide.columns]
    df_wide   = df_wide[["image_path"] + tab_features + available].dropna()

    print(f"Dataset: {len(df_wide)} images")
    print(f"Targets: {available}")
    print(f"Tabular features: {tab_features}")
    print(f"NDVI range: {df_wide['Pre_GSHH_NDVI'].min():.2f} – {df_wide['Pre_GSHH_NDVI'].max():.2f}")
    print(f"Height range: {df_wide['Height_Ave_cm'].min():.2f} – {df_wide['Height_Ave_cm'].max():.2f} cm")

    # train/val split
    train_df, val_df = train_test_split(
        df_wide, test_size=val_split, random_state=seed
    )
    print(f"Train: {len(train_df)} | Val: {len(val_df)}")

    # fit scaler ONLY on train set
    scaler = StandardScaler()
    scaler.fit(train_df[tab_features].values)

    return train_df, val_df, available, scaler

class BiomassDatasetTabular(Dataset):
    """
    Returns (image, tabular_features, targets) for each sample.
    tabular_features: scaled [NDVI, Height] as a float tensor
    """
    def __init__(self, df_wide, image_dir, targets, tab_features,
                 scaler, transform=None):
        self.df          = df_wide.reset_index(drop=True)
        self.image_dir   = image_dir
        self.targets     = targets
        self.tab_features= tab_features
        self.scaler      = scaler
        self.transform   = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row      = self.df.iloc[idx]

        # ── image ──────────────────────────────
        img_path = os.path.join(self.image_dir, row["image_path"])
        image    = Image.open(img_path).convert("RGB")
        if self.transform:
            image = self.transform(image)

        # ── tabular ────────────────────────────
        tab_raw  = row[self.tab_features].values.astype(np.float32)
        tab_scaled = self.scaler.transform(tab_raw.reshape(1, -1))[0]
        tab_tensor = torch.tensor(tab_scaled, dtype=torch.float32)

        # ── labels ─────────────────────────────
        labels   = torch.tensor(
            row[self.targets].values.astype(np.float32),
            dtype=torch.float32
        )
        return image, tab_tensor, labels

# same as before    
def get_transforms(image_size, mode="train"):
    mean = [0.485, 0.456, 0.406]
    std  = [0.229, 0.224, 0.225]

    if mode == "train":
        return transforms.Compose([
            transforms.Resize((image_size + 32, image_size + 32)),
            transforms.RandomCrop(image_size),
            transforms.RandomHorizontalFlip(),
            transforms.RandomVerticalFlip(),
            transforms.ColorJitter(brightness=0.3, contrast=0.3,
                                   saturation=0.3, hue=0.1),
            transforms.RandomRotation(30),
            transforms.ToTensor(),
            transforms.Normalize(mean, std),
        ])
    else:
        return transforms.Compose([
            transforms.Resize((image_size, image_size)),
            transforms.ToTensor(),
            transforms.Normalize(mean, std),
        ])
    

cfg = Config()
torch.manual_seed(cfg.SEED)
np.random.seed(cfg.SEED)
print(f"Device: {cfg.DEVICE}")

# ── 1. Data ───────────────────────────────
train_df, val_df, targets, scaler = prepare_data(
    cfg.CSV_PATH, cfg.TARGETS, cfg.TABULAR_FEATURES,
    cfg.VAL_SPLIT, cfg.SEED
)
joblib.dump(scaler, cfg.SCALER_PATH)
print(f"Scaler saved → {cfg.SCALER_PATH}")

train_dataset = BiomassDatasetTabular(
    train_df, cfg.IMAGE_DIR, targets, cfg.TABULAR_FEATURES,
    scaler, transform=get_transforms(cfg.IMAGE_SIZE, "train")
)
val_dataset = BiomassDatasetTabular(
    val_df, cfg.IMAGE_DIR, targets, cfg.TABULAR_FEATURES,
    scaler, transform=get_transforms(cfg.IMAGE_SIZE, "val")
)

train_loader = DataLoader(train_dataset, batch_size=cfg.BATCH_SIZE,
                            shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_dataset,   batch_size=cfg.BATCH_SIZE,
                            shuffle=False, num_workers=2, pin_memory=True)


Device: cuda
Dataset: 357 images
Targets: ['Dry_Green_g', 'GDM_g', 'Dry_Total_g']
Tabular features: ['Pre_GSHH_NDVI', 'Height_Ave_cm']
NDVI range: 0.16 – 0.91
Height range: 1.00 – 70.00 cm
Train: 285 | Val: 72
Scaler saved → model2_scaler.pkl


In [17]:
class BiomassModelTabular(nn.Module):
    """
    Two-branch fusion model:
      Branch 1: Pretrained CNN → image embedding
      Branch 2: Small MLP     → tabular embedding
      Fusion:   Concatenate both embeddings → regression head → 5 outputs

    Why this architecture?
      - The image branch learns visual texture/color patterns
      - The tabular branch encodes physical measurements (NDVI = greenness,
        Height = biomass proxy) that directly correlate with targets
      - Concatenation lets the head learn to combine both signals freely
    """
    def __init__(self, backbone_name, num_tabular, num_targets,
                 pretrained=True, dropout=0.3, tab_hidden=32):
        super().__init__()

        # ── image backbone ────────────────────
        if backbone_name == "efficientnet_b0":
            weights  = models.EfficientNet_B0_Weights.DEFAULT if pretrained else None
            backbone = models.efficientnet_b0(weights=weights)
            img_features = backbone.classifier[1].in_features
            backbone.classifier = nn.Identity()

        elif backbone_name == "resnet50":
            weights  = models.ResNet50_Weights.DEFAULT if pretrained else None
            backbone = models.resnet50(weights=weights)
            img_features = backbone.fc.in_features
            backbone.fc = nn.Identity()

        elif backbone_name == "resnet34":
            weights  = models.ResNet34_Weights.DEFAULT if pretrained else None
            backbone = models.resnet34(weights=weights)
            img_features = backbone.fc.in_features
            backbone.fc = nn.Identity()

        else:
            raise ValueError(f"Unknown backbone: {backbone_name}")

        self.image_branch = backbone

        # ── tabular branch ────────────────────
        # Small MLP to embed [NDVI, Height] into a richer representation
        self.tab_branch = nn.Sequential(
            nn.Linear(num_tabular, tab_hidden),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(tab_hidden, tab_hidden),
            nn.ReLU(),
        )

        # ── fusion regression head ────────────
        fused_size = img_features + tab_hidden
        self.head = nn.Sequential(
            nn.Linear(fused_size, 256),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(128, num_targets),
            nn.ReLU()          # biomass >= 0
        )

    def forward(self, image, tabular):
        img_emb = self.image_branch(image)          # (B, img_features)
        tab_emb = self.tab_branch(tabular)           # (B, tab_hidden)
        fused   = torch.cat([img_emb, tab_emb], dim=1)  # (B, img_features + tab_hidden)
        return self.head(fused)
    
def train_one_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss = 0.0
    loss_weights = torch.tensor([1.0, 1.0, 2.5]).to(device)

    for images, tabular, labels in loader:
        images  = images.to(device)
        tabular = tabular.to(device)
        labels  = labels.to(device)

        optimizer.zero_grad()
        preds = model(images, tabular)
        # loss  = criterion(preds, labels)
        loss  = (loss_weights * ((preds - labels) ** 2)).mean()
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * images.size(0)

    return total_loss / len(loader.dataset)


@torch.no_grad()
def evaluate(model, loader, criterion, device, targets):
    model.eval()
    total_loss = 0.0
    loss_weights = torch.tensor([1.0, 1.0, 2.5]).to(device)
    all_preds  = []
    all_labels = []

    for images, tabular, labels in loader:
        images  = images.to(device)
        tabular = tabular.to(device)
        labels  = labels.to(device)

        preds = model(images, tabular)
        # loss  = criterion(preds, labels)
        loss  = (loss_weights * ((preds - labels) ** 2)).mean()
        total_loss += loss.item() * images.size(0)
        all_preds.append(preds.cpu().numpy())
        all_labels.append(labels.cpu().numpy())

    all_preds  = np.vstack(all_preds)
    all_labels = np.vstack(all_labels)

    r2_scores = {}
    for i, t in enumerate(targets):
        ss_res = np.sum((all_labels[:, i] - all_preds[:, i]) ** 2)
        ss_tot = np.sum((all_labels[:, i] - all_labels[:, i].mean()) ** 2)
        r2_scores[t] = 1 - ss_res / (ss_tot + 1e-8)

    return total_loss / len(loader.dataset), r2_scores, all_preds, all_labels


# ─────────────────────────────────────────────
# PLOTTING
# ─────────────────────────────────────────────
def plot_training_curves(train_losses, val_losses,
                         save_path="model2_training_curves.png"):
    plt.figure(figsize=(8, 4))
    plt.plot(train_losses, label="Train Loss")
    plt.plot(val_losses,   label="Val Loss")
    plt.xlabel("Epoch")
    plt.ylabel("MSE Loss")
    plt.title("Model 2 (Image + Tabular) Training Curves")
    plt.legend()
    plt.tight_layout()
    plt.savefig(save_path)
    plt.close()
    print(f"Saved training curves → {save_path}")


def plot_predictions(all_preds, all_labels, targets,
                     save_path="model2_predictions.png"):
    n = len(targets)
    fig, axes = plt.subplots(1, n, figsize=(4 * n, 4))
    if n == 1:
        axes = [axes]

    for i, (ax, t) in enumerate(zip(axes, targets)):
        ax.scatter(all_labels[:, i], all_preds[:, i], alpha=0.5, s=20)
        lims = [
            min(all_labels[:, i].min(), all_preds[:, i].min()),
            max(all_labels[:, i].max(), all_preds[:, i].max()),
        ]
        ax.plot(lims, lims, "r--", linewidth=1)
        ax.set_xlabel("Ground Truth (g)")
        ax.set_ylabel("Predicted (g)")
        ax.set_title(t)

    plt.suptitle("Model 2: Image + Tabular Predictions vs Ground Truth", y=1.02)
    plt.tight_layout()
    plt.savefig(save_path, bbox_inches="tight")
    plt.close()
    print(f"Saved prediction plots → {save_path}")

# ─────────────────────────────────────────────
# INFERENCE
# ─────────────────────────────────────────────
def predict(model, image_path, ndvi, height_cm,
            targets, tab_features, scaler, image_size, device):
    """
    Run inference on a single image + tabular inputs.
    
    Args:
        image_path : path to the pasture image
        ndvi       : Pre_GSHH_NDVI reading (float)
        height_cm  : Height_Ave_cm reading (float)
    
    Returns:
        dict {target_name: predicted_grams}
    """
    transform = get_transforms(image_size, mode="val")
    image     = Image.open(image_path).convert("RGB")
    image     = transform(image).unsqueeze(0).to(device)

    tab_raw    = np.array([[ndvi, height_cm]], dtype=np.float32)
    tab_scaled = scaler.transform(tab_raw)
    tab_tensor = torch.tensor(tab_scaled, dtype=torch.float32).to(device)

    model.eval()
    with torch.no_grad():
        preds = model(image, tab_tensor).cpu().numpy()[0]

    result = {t: round(float(v), 4) for t, v in zip(targets, preds)}

    # simple grazing decision on top of prediction
    total = result.get("Dry_Total_g", 0)
    GRAZING_THRESHOLD = 60.0   # grams — adjust based on domain knowledge
    result["grazing_ready"] = total >= GRAZING_THRESHOLD
    result["recommendation"] = (
        f"Total biomass: {total:.1f}g. "
        + ("✅ Ready for grazing." if total >= GRAZING_THRESHOLD
           else f"⚠️ Below threshold ({GRAZING_THRESHOLD}g). Wait before grazing.")
    )
    return result


In [18]:
model = BiomassModelTabular(
    backbone_name=cfg.BACKBONE,
    num_tabular=len(cfg.TABULAR_FEATURES),
    num_targets=len(targets),
    pretrained=cfg.PRETRAINED,
    dropout=cfg.DROPOUT,
    tab_hidden=cfg.TAB_HIDDEN,
).to(cfg.DEVICE)

print(f"\nModel: {cfg.BACKBONE} + tabular{cfg.TABULAR_FEATURES}")
print(f"Targets: {targets}")

# ── 3. Training setup ─────────────────────
# criterion = nn.MSELoss()
optimizer = torch.optim.AdamW(
    model.parameters(), lr=cfg.LR, weight_decay=cfg.WEIGHT_DECAY
)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=cfg.NUM_EPOCHS, eta_min=1e-6
)

# ── 4. Training loop ──────────────────────
train_losses, val_losses = [], []
best_val_loss = float("inf")

for epoch in range(1, cfg.NUM_EPOCHS + 1):
    train_loss = train_one_epoch(
        model, train_loader, optimizer, criterion, cfg.DEVICE
    )
    val_loss, r2_scores, val_preds, val_labels = evaluate(
        model, val_loader, criterion, cfg.DEVICE, targets
    )
    scheduler.step()

    train_losses.append(train_loss)
    val_losses.append(val_loss)

    if epoch % 5 == 0 or epoch == 1:
        r2_str = " | ".join([f"{t}: {v:.3f}" for t, v in r2_scores.items()])
        print(f"Epoch {epoch:3d}/{cfg.NUM_EPOCHS} | "
                f"Train: {train_loss:.4f} | Val: {val_loss:.4f}")
        print(f"  R² → {r2_str}")

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save({
            "epoch":        epoch,
            "model_state":  model.state_dict(),
            "optimizer":    optimizer.state_dict(),
            "targets":      targets,
            "tab_features": cfg.TABULAR_FEATURES,
            "config": {
                "backbone":   cfg.BACKBONE,
                "image_size": cfg.IMAGE_SIZE,
                "dropout":    cfg.DROPOUT,
                "tab_hidden": cfg.TAB_HIDDEN,
            }
        }, cfg.SAVE_PATH)

print(f"\nBest val loss: {best_val_loss:.4f} → saved to {cfg.SAVE_PATH}")

# ── 5. Final evaluation ───────────────────
checkpoint = torch.load(cfg.SAVE_PATH, map_location=cfg.DEVICE)
model.load_state_dict(checkpoint["model_state"])
_, r2_final, val_preds, val_labels = evaluate(
    model, val_loader, criterion, cfg.DEVICE, targets
)

print("\n── Final Val R² Scores (Model 2) ──")
for t, r2 in r2_final.items():
    print(f"  {t:<20s}: {r2:.4f}")

# ── 6. Plots ──────────────────────────────
plot_training_curves(train_losses, val_losses)
plot_predictions(val_preds, val_labels, targets)


Model: resnet34 + tabular['Pre_GSHH_NDVI', 'Height_Ave_cm']
Targets: ['Dry_Green_g', 'GDM_g', 'Dry_Total_g']
Epoch   1/150 | Train: 3441.5927 | Val: 2758.2323
  R² → Dry_Green_g: -1.145 | GDM_g: -2.007 | Dry_Total_g: -2.998
Epoch   5/150 | Train: 1472.0850 | Val: 1012.6773
  R² → Dry_Green_g: -1.145 | GDM_g: -0.590 | Dry_Total_g: 0.131
Epoch  10/150 | Train: 576.1703 | Val: 379.7754
  R² → Dry_Green_g: 0.472 | GDM_g: 0.590 | Dry_Total_g: 0.523
Epoch  15/150 | Train: 514.0936 | Val: 680.4864
  R² → Dry_Green_g: 0.261 | GDM_g: 0.118 | Dry_Total_g: 0.128
Epoch  20/150 | Train: 402.4832 | Val: 481.2750
  R² → Dry_Green_g: 0.531 | GDM_g: 0.417 | Dry_Total_g: 0.352
Epoch  25/150 | Train: 417.4929 | Val: 339.7577
  R² → Dry_Green_g: 0.596 | GDM_g: 0.621 | Dry_Total_g: 0.555
Epoch  30/150 | Train: 283.2558 | Val: 269.1502
  R² → Dry_Green_g: 0.696 | GDM_g: 0.709 | Dry_Total_g: 0.640
Epoch  35/150 | Train: 274.5958 | Val: 254.3065
  R² → Dry_Green_g: 0.681 | GDM_g: 0.678 | Dry_Total_g: 0.685
E